# OFFIRST final contrastive-distilled retinal encoder

Controlled continuation of KD768. No DR classifier, no ONNX export, and no Android change in this notebook.

Run the cells in order on a Kaggle GPU session. Stop at the first FAIL. Cell 3 is not included until Cell 1 and Cell 2 have been inspected.

## Cell 1 — Environment and reproducibility

Records the Python, PyTorch, and GPU versions and sets the experiment seed. It does not read checkpoints or images.

**PASS:** PyTorch imports, CUDA is available, and `/kaggle/input` exists.

**FAIL:** no GPU, PyTorch missing, or this cell is running outside a Kaggle session.

In [ ]:
import json
import os
import platform
import random
import sys
from pathlib import Path

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)

def module_version(name):
    try:
        module = __import__(name)
        return getattr(module, "__version__", "present")
    except Exception as error:
        return f"MISSING: {type(error).__name__}: {error}"

report = {
    "cell": 1,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "seed": SEED,
}
for name in ("torch", "torchvision", "timm", "numpy", "cv2", "PIL", "sklearn", "pandas"):
    report[name] = module_version(name)

import numpy as np
np.random.seed(SEED)

cuda_ok = False
if not str(report["torch"]).startswith("MISSING"):
    import torch
    torch.manual_seed(SEED)
    cuda_ok = bool(torch.cuda.is_available())
    report["torch_cuda"] = torch.version.cuda
    report["cudnn"] = torch.backends.cudnn.version()
    report["gpu_count"] = torch.cuda.device_count()
    if cuda_ok:
        torch.cuda.manual_seed_all(SEED)
        report["gpus"] = []
        for index in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(index)
            report["gpus"].append({
                "index": index,
                "name": props.name,
                "total_gb": round(props.total_memory / 2**30, 2),
                "capability": f"{props.major}.{props.minor}",
            })

report["kaggle_input_exists"] = Path("/kaggle/input").is_dir()
report["kaggle_working_exists"] = Path("/kaggle/working").is_dir()

failures = []
if str(report["torch"]).startswith("MISSING"):
    failures.append("PyTorch is not installed.")
if not cuda_ok:
    failures.append("CUDA is not available. Use a Kaggle GPU session before any training cell.")
if not report["kaggle_input_exists"]:
    failures.append("/kaggle/input is missing. Run this notebook on Kaggle with the datasets attached.")
for name in ("timm", "numpy", "cv2", "sklearn", "pandas"):
    if str(report[name]).startswith("MISSING"):
        failures.append(f"{name} is not installed.")

report["gate"] = "PASS" if not failures else "FAIL"
report["failures"] = failures
print(json.dumps(report, indent=2))
assert report["gate"] == "PASS", failures


## Cell 2 — Input and checkpoint discovery

Finds the mounted files. It does not load weights, hash them, or train.

The only acceptable student start is `kd_run2_kd_final.pth`. `student_init_mnv3_ssl_ep40_target.pth` is the pre-KD backbone and is recorded only so it cannot be selected by mistake. The existing weights dataset `teamoffirst/offirst-kd-run2-weights` contains the teacher and that pre-KD init, not the KD768 final checkpoint.

**PASS:** exactly one teacher file and exactly one `kd_run2_kd_final.pth` under `/kaggle/input`.

**FAIL:** either required file is missing or duplicated. A missing image dataset is reported separately and does not hide a checkpoint failure.

In [ ]:
import json
from pathlib import Path

INPUT = Path("/kaggle/input")
REQUIRED = {
    "teacher": "teacher_byol_v4_ep20_target_convnext_tiny.pth",
    "student_kd768": "kd_run2_kd_final.pth",
}
EXPECTED_SHA256 = {
    "teacher": "ed95b4de631bbeee1776775ec8d309313bcdd142de03609ab9c8185e40cde784",
    "student_kd768": "8bc239d22ffc4e786820f2151c99380a754bef930e42edea65022e1246f2096d",
}
NOT_A_START = (
    "student_init_mnv3_ssl_ep40_target.pth",
    "kd_run2_last_pre_recal.pth",
    "kd_run2_smoke_v2_kd_final.pth",
    "kd_run2_localtest_kd_final.pth",
    "kd_run2_ep010.pth",
    "kd_run2_ep020.pth",
    "kd_run2_ep030.pth",
)
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
TABLE_SUFFIXES = {".csv", ".xlsx", ".xls", ".json"}

def locate(filename):
    return sorted(path for path in INPUT.rglob(filename) if path.is_file())

def describe(path):
    parts = path.relative_to(INPUT).parts
    dataset = "/".join(parts[:2]) if len(parts) >= 2 else parts[0]
    return {
        "path": str(path),
        "dataset": dataset,
        "bytes": path.stat().st_size,
    }

found = {role: [describe(path) for path in locate(name)] for role, name in REQUIRED.items()}
decoys = {name: [describe(path) for path in locate(name)] for name in NOT_A_START}
decoys = {name: rows for name, rows in decoys.items() if rows}

datasets = []
if INPUT.is_dir():
    owners = sorted(path for path in INPUT.iterdir() if path.is_dir())
    dataset_dirs = []
    for owner in owners:
        children = [path for path in owner.iterdir() if path.is_dir()]
        dataset_dirs.extend(children or [owner])
    for root in dataset_dirs:
        image_count = 0
        tables = []
        samples = []
        for path in root.rglob("*"):
            if not path.is_file():
                continue
            suffix = path.suffix.lower()
            if suffix in IMAGE_SUFFIXES:
                image_count += 1
                if len(samples) < 3:
                    samples.append(str(path.relative_to(root)))
            elif suffix in TABLE_SUFFIXES and len(tables) < 12:
                tables.append(str(path.relative_to(root)))
        slug = str(root.relative_to(INPUT)).lower()
        if "rfmid" in slug:
            kind = "rfmid_candidate"
        elif "odir" in slug or "ocular" in slug:
            kind = "odir_candidate"
        else:
            kind = "unclassified"
        datasets.append({
            "dataset": str(root.relative_to(INPUT)),
            "kind": kind,
            "image_count": image_count,
            "tables": tables,
            "sample_images": samples,
        })

failures = []
for role, name in REQUIRED.items():
    rows = found[role]
    if len(rows) == 0:
        failures.append(f"Missing {name}. Attach the private dataset that contains this exact file.")
    elif len(rows) > 1:
        failures.append(f"Found {len(rows)} copies of {name}. Keep one mounted copy before Cell 3.")

data_notes = []
if not any(row["kind"] == "rfmid_candidate" and row["image_count"] > 0 for row in datasets):
    data_notes.append("No mounted dataset slug contains 'rfmid' with images. Paste this report before treating a folder as RFMiD.")
if not any(row["kind"] == "odir_candidate" and row["image_count"] > 0 for row in datasets):
    data_notes.append("No mounted dataset slug contains 'odir' or 'ocular' with images. Paste this report before treating a folder as ODIR.")

report = {
    "cell": 2,
    "required": found,
    "expected_sha256_for_cell_3": EXPECTED_SHA256,
    "present_but_not_a_student_start": decoys,
    "datasets": datasets,
    "checkpoint_gate": "PASS" if not failures else "FAIL",
    "checkpoint_failures": failures,
    "dataset_notes": data_notes,
}
print(json.dumps(report, indent=2))
assert report["checkpoint_gate"] == "PASS", failures
